In [21]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import requests

# Localizar o projeto, executando da raiz ou de uma subpasta.
diretorio_atual = Path.cwd().resolve()

ROOT = next(
    (
        pasta
        for pasta in (diretorio_atual, *diretorio_atual.parents)
        if (pasta / "code" / "sensores.json").is_file()
    ),
    None,
)

if ROOT is None:
    raise RuntimeError(
        "Não foi possível localizar o projeto. "
        "Execute o notebook dentro de temperatureMonitoring."
    )

BASE_DIR = ROOT / "code" / "tcc"

RAW_DIR = BASE_DIR / "data" / "raw" / "questdb"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
METADATA_DIR = BASE_DIR / "data" / "metadata"

for pasta in (RAW_DIR, PROCESSED_DIR, METADATA_DIR):
    pasta.mkdir(parents=True, exist_ok=True)

BASE_URL = "http://andromeda.lasdpc.icmc.usp.br:61102/questdb"

SENSOR_IDS = [
    1, 48, 887, 891, 86782,
    86789, 86790, 86791, 86793,
]

INICIO = pd.Timestamp("2026-06-10T00:00:00Z")
FIM = pd.Timestamp("2026-07-26T00:00:00Z")  # Exclusivo
FREQUENCIA = "5min"

ARQUIVO_BRUTO = (
    RAW_DIR / "temperaturas_1006_2026-06-10_2026-07-26_5min.csv"
)

# False: reutilizar a extração local, se já existir.
# True: consultar o banco e atualizar a extração.
ATUALIZAR_EXTRACAO = False

print("Projeto:", ROOT)
print("Período:", INICIO, "até", FIM, "(fim exclusivo)")

Projeto: /home/gustavomoura/Documents/temperatureMonitoring
Período: 2026-06-10 00:00:00+00:00 até 2026-07-26 00:00:00+00:00 (fim exclusivo)


In [22]:
ids_sql = ", ".join(f"'{sensor_id}'" for sensor_id in SENSOR_IDS)

SQL = f"""
SELECT
    measured_in,
    sensor_id,
    avg(temperature) AS temperatura_media,
    count(*) AS quantidade
FROM sensor_temperature
WHERE environment = '1006'
  AND measured_in >= '{INICIO.strftime("%Y-%m-%dT%H:%M:%SZ")}'
  AND measured_in < '{FIM.strftime("%Y-%m-%dT%H:%M:%SZ")}'
  AND sensor_id IN ({ids_sql})
SAMPLE BY 5m
ORDER BY measured_in, sensor_id
"""

print(SQL)


SELECT
    measured_in,
    sensor_id,
    avg(temperature) AS temperatura_media,
    count(*) AS quantidade
FROM sensor_temperature
WHERE environment = '1006'
  AND measured_in >= '2026-06-10T00:00:00Z'
  AND measured_in < '2026-07-26T00:00:00Z'
  AND sensor_id IN ('1', '48', '887', '891', '86782', '86789', '86790', '86791', '86793')
SAMPLE BY 5m
ORDER BY measured_in, sensor_id



In [23]:
if ARQUIVO_BRUTO.exists() and not ATUALIZAR_EXTRACAO:
    temperaturas = pd.read_csv(ARQUIVO_BRUTO)
    print("Extração local carregada:", ARQUIVO_BRUTO)

else:
    resposta = requests.get(
        f"{BASE_URL}/exec",
        params={
            "query": SQL,
            "limit": "0,200000",
        },
        timeout=180,
    )
    resposta.raise_for_status()

    dados = resposta.json()

    if "error" in dados:
        raise RuntimeError(f"Erro do QuestDB: {dados['error']}")

    if "columns" not in dados or "dataset" not in dados:
        raise RuntimeError("Resposta sem columns ou dataset.")

    temperaturas = pd.DataFrame(
        dados["dataset"],
        columns=[coluna["name"] for coluna in dados["columns"]],
    )

    if temperaturas.empty:
        raise RuntimeError("A consulta não retornou registros.")

    if dados.get("count", len(temperaturas)) > len(temperaturas):
        raise RuntimeError(
            "A resposta parece truncada. "
            "Não utilizar esta extração antes de conferir a paginação."
        )

    temperaturas.to_csv(ARQUIVO_BRUTO, index=False)

    ARQUIVO_BRUTO.with_suffix(".sql").write_text(
        SQL,
        encoding="utf-8",
    )

    metadados_extracao = {
        "extraido_em_utc": pd.Timestamp.now(tz="UTC").isoformat(),
        "endpoint": f"{BASE_URL}/exec",
        "inicio_inclusivo": INICIO.isoformat(),
        "fim_exclusivo": FIM.isoformat(),
        "sensor_ids": SENSOR_IDS,
        "linhas_retornadas": len(temperaturas),
        "sql": SQL,
        "sha256_csv": hashlib.sha256(
            ARQUIVO_BRUTO.read_bytes()
        ).hexdigest(),
    }

    ARQUIVO_BRUTO.with_suffix(".json").write_text(
        json.dumps(
            metadados_extracao,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    print("Extração salva:", ARQUIVO_BRUTO)

print("Linhas da extração:", len(temperaturas))
display(temperaturas.head())

Extração local carregada: /home/gustavomoura/Documents/temperatureMonitoring/code/tcc/data/raw/questdb/temperaturas_1006_2026-06-10_2026-07-26_5min.csv
Linhas da extração: 116438


,measured_in,sensor_id,temperatura_media,quantidade
0,2026-06-10T01:40:00.000000Z,1,22.725,4
1,2026-06-10T01:40:00.000000Z,48,23.100,6
2,2026-06-10T01:40:00.000000Z,86782,22.650,4
3,2026-06-10T01:40:00.000000Z,86789,22.500,2
4,2026-06-10T01:40:00.000000Z,86790,24.700,2


In [24]:
colunas_obrigatorias = {
    "measured_in",
    "sensor_id",
    "temperatura_media",
    "quantidade",
}

if not colunas_obrigatorias.issubset(temperaturas.columns):
    raise ValueError(
        f"Colunas inesperadas: {temperaturas.columns.tolist()}"
    )

temperaturas["measured_in"] = pd.to_datetime(
    temperaturas["measured_in"],
    utc=True,
    errors="raise",
)

for coluna in ("sensor_id", "temperatura_media", "quantidade"):
    temperaturas[coluna] = pd.to_numeric(
        temperaturas[coluna],
        errors="raise",
    )

if temperaturas[["measured_in", "sensor_id", "quantidade"]].isna().any().any():
    raise ValueError("Há timestamps, IDs ou contagens ausentes.")

if not temperaturas["sensor_id"].isin(SENSOR_IDS).all():
    raise ValueError("Há sensores fora do conjunto experimental.")

sensores_ausentes = set(SENSOR_IDS) - set(temperaturas["sensor_id"])

if sensores_ausentes:
    raise ValueError(f"Sensores sem registros: {sensores_ausentes}")

if not temperaturas["quantidade"].ge(1).all():
    raise ValueError("Há janelas com contagem inválida.")

dentro_periodo = (
    temperaturas["measured_in"].ge(INICIO)
    & temperaturas["measured_in"].lt(FIM)
)

if not dentro_periodo.all():
    raise ValueError("Há timestamps fora do período solicitado.")

na_grade = temperaturas["measured_in"].eq(
    temperaturas["measured_in"].dt.floor(FREQUENCIA)
)

if not na_grade.all():
    raise ValueError(
        "Há timestamps fora da grade de cinco minutos. "
        "Verifique o alinhamento da agregação."
    )

duplicados = temperaturas.duplicated(
    subset=["measured_in", "sensor_id"],
    keep=False,
)

if duplicados.any():
    display(temperaturas.loc[duplicados].head(20))
    raise ValueError("Há mais de uma linha por sensor e janela.")

valores_presentes = temperaturas["temperatura_media"].dropna()

if not np.isfinite(valores_presentes.to_numpy()).all():
    raise ValueError("Há temperaturas infinitas.")

temperaturas = temperaturas.sort_values(
    ["measured_in", "sensor_id"]
).reset_index(drop=True)

print("Validação estrutural concluída.")
print(
    "Temperaturas médias ausentes na extração:",
    temperaturas["temperatura_media"].isna().sum(),
)

Validação estrutural concluída.
Temperaturas médias ausentes na extração: 0


In [25]:
indice_completo = pd.date_range(
    start=INICIO,
    end=FIM,
    freq=FREQUENCIA,
    inclusive="left",
    name="timestamp",
)

matriz = temperaturas.pivot(
    index="measured_in",
    columns="sensor_id",
    values="temperatura_media",
).reindex(
    index=indice_completo,
    columns=SENSOR_IDS,
)

contagens = temperaturas.pivot(
    index="measured_in",
    columns="sensor_id",
    values="quantidade",
).reindex(
    index=indice_completo,
    columns=SENSOR_IDS,
)

matriz.columns.name = "sensor_id"
contagens.columns.name = "sensor_id"

resumo_sensores = pd.DataFrame({
    "sensor_id": SENSOR_IDS,
    "janelas_esperadas": len(matriz),
    "janelas_validas": matriz.notna().sum().to_numpy(),
    "janelas_ausentes": matriz.isna().sum().to_numpy(),
    "cobertura_pct": (
        100 * matriz.notna().mean()
    ).to_numpy(),
    "media_c": matriz.mean().to_numpy(),
    "desvio_c": matriz.std().to_numpy(),
    "min_c": matriz.min().to_numpy(),
    "max_c": matriz.max().to_numpy(),
})

dados_completos = matriz.dropna(how="any").copy()

if dados_completos.empty:
    raise RuntimeError("Não há janelas completas para os nove sensores.")

print("Matriz com todas as janelas:", matriz.shape)
print("Matriz somente com janelas completas:", dados_completos.shape)
print(
    "Janelas completas (%):",
    round(100 * len(dados_completos) / len(matriz), 2),
)
print("Primeira janela completa:", dados_completos.index.min())
print("Última janela completa:", dados_completos.index.max())

display(resumo_sensores.round(3))
display(dados_completos.head())

Matriz com todas as janelas: (13248, 9)
Matriz somente com janelas completas: (12892, 9)
Janelas completas (%): 97.31
Primeira janela completa: 2026-06-10 01:40:00+00:00
Última janela completa: 2026-07-25 09:55:00+00:00


,sensor_id,janelas_esperadas,janelas_validas,janelas_ausentes,cobertura_pct,media_c,desvio_c,min_c,max_c
0,1,13248,12944,304,97.705,21.994,1.185,18.542,25.100
1,48,13248,12902,346,97.388,22.113,0.993,19.300,24.800
2,887,13248,12931,317,97.607,22.617,1.154,19.609,25.800
3,891,13248,12954,294,97.781,19.847,0.805,18.000,22.600
4,86782,13248,12963,285,97.849,22.002,1.187,18.400,26.254
5,86789,13248,12918,330,97.509,21.551,1.172,18.409,25.000
6,86790,13248,12925,323,97.562,24.019,1.155,20.182,27.320
7,86791,13248,12941,307,97.683,23.275,1.168,19.180,26.689
8,86793,13248,12960,288,97.826,22.494,1.277,19.000,26.818


sensor_id,1,48,887,891,86782,86789,86790,86791,86793
timestamp,,,,,,,,,
2026-06-10 01:40:00+00:00,22.725,23.100,23.740000,20.400000,22.650000,22.500000,24.700000,24.000,23.600000
2026-06-10 01:45:00+00:00,22.700,23.100,23.700000,20.400000,22.600000,22.428571,24.600000,23.875,23.600000
2026-06-10 01:50:00+00:00,22.600,23.000,23.687500,20.322222,22.545455,22.400000,24.533333,23.810,23.500000
2026-06-10 01:55:00+00:00,22.600,23.025,23.671429,20.400000,22.500000,22.325000,24.500000,23.800,23.500000
2026-06-10 02:00:00+00:00,22.600,23.090,23.628571,20.400000,22.525000,22.300000,24.500000,23.780,23.463636


In [26]:
comparacao = pd.DataFrame([
    {
        "indicador": "janelas_totais",
        "anterior": 13248,
        "atual": len(matriz),
    },
    {
        "indicador": "janelas_completas",
        "anterior": 12892,
        "atual": len(dados_completos),
    },
])

comparacao["confere"] = (
    comparacao["anterior"] == comparacao["atual"]
)

display(comparacao)

matriz.to_csv(
    PROCESSED_DIR / "temperaturas_1006_5min.csv",
    index_label="timestamp",
)

dados_completos.to_csv(
    PROCESSED_DIR / "temperaturas_1006_5min_completas.csv",
    index_label="timestamp",
)

contagens.to_csv(
    PROCESSED_DIR / "contagens_1006_5min.csv",
    index_label="timestamp",
)

resumo_sensores.to_csv(
    METADATA_DIR / "resumo_questdb_1006.csv",
    index=False,
)

comparacao.to_csv(
    METADATA_DIR / "comparacao_contagens_questdb.csv",
    index=False,
)

auditoria = {
    "sha256_extracao_csv": hashlib.sha256(
        ARQUIVO_BRUTO.read_bytes()
    ).hexdigest(),
    "sensor_ids": SENSOR_IDS,
    "inicio_inclusivo_utc": INICIO.isoformat(),
    "fim_exclusivo_utc": FIM.isoformat(),
    "frequencia": FREQUENCIA,
    "tratamento_ausentes": "Exclusão de janelas com qualquer sensor ausente",
    "imputacao": False,
    "filtro_minimo_medicoes_por_janela": None,
    "janelas_totais": len(matriz),
    "janelas_completas": len(dados_completos),
    "janelas_excluidas": len(matriz) - len(dados_completos),
    "primeira_janela_completa": dados_completos.index.min().isoformat(),
    "ultima_janela_completa": dados_completos.index.max().isoformat(),
    "contagens_anteriores_conferem": bool(comparacao["confere"].all()),
}

(METADATA_DIR / "auditoria_preprocessamento_1006.json").write_text(
    json.dumps(auditoria, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Arquivos exportados para:", PROCESSED_DIR)

if not comparacao["confere"].all():
    print(
        "ATENÇÃO: as contagens diferem das saídas anteriores. "
        "Investigue a extração antes de executar os modelos."
    )

,indicador,anterior,atual,confere
0,janelas_totais,13248,13248,True
1,janelas_completas,12892,12892,True


Arquivos exportados para: /home/gustavomoura/Documents/temperatureMonitoring/code/tcc/data/processed
